# 10 — Otimização temporal do HGB com Optuna

**Estudo concluído no relato do autor:** 20 trials, 63 fits, nenhuma falha e
`retain_reference`. Nenhum trial passou o gate. **HGB = Histogram-based Gradient
Boosting**, modelo de árvores que permanece como referência.

Este notebook revisa artefatos salvos; não inicia treinamento nem consulta setembro.
O [resultado da busca](../../docs/modeling/EVALUATION_PROTOCOL.md#resultado-da-busca-informado-pelo-autor)
registra a evidência e seus limites. O orçamento está encerrado; os comandos de
preparação/execução continuam documentados para o protocolo correspondente, sem
autorizar repetir ou ampliar esse estudo.


In [ ]:
import json
from pathlib import Path

import pandas as pd

from fraud_detection_mlops.config import PROJECT_ROOT
from fraud_detection_mlops.modeling.experiments import (
    hgb_optimization as hgb_optuna,
)
from fraud_detection_mlops.modeling.experiments import (
    temporal_development_data as development,
)

# Substitua pelo development_path retornado por hgb_optuna prepare.
DEVELOPMENT_PATH = Path("CAMINHO_RETORNADO")
if not DEVELOPMENT_PATH.is_absolute():
    DEVELOPMENT_PATH = PROJECT_ROOT / DEVELOPMENT_PATH
STUDY_PATH = DEVELOPMENT_PATH / "study"
result = hgb_optuna.verify_hgb_optimization(STUDY_PATH)
policy = development.load_policy()
result

## Comparação pareada e custos

AP média é o objetivo dos três cortes. O gate considera ganho médio de AP e
precisão @100, além da perda em cada corte. O estudo concluído não produziu
candidato elegível à confirmação; a seleção não demonstra superioridade.

RSS é o pico acumulado do processo, não memória isolada de cada fit. Custos e
resultados por corte são inspecionados nos recibos salvos abaixo.


In [ ]:
reference = pd.DataFrame([result["reference"]])
comparison = pd.DataFrame(result["comparison"])
display(reference)
display(comparison)

In [ ]:
rows = []
for receipt in sorted((STUDY_PATH / "models").glob("*/*/result.json")):
    record = json.loads(receipt.read_text())
    rows.append(
        {
            "candidate": receipt.parent.parent.name,
            "fold": record["fold_id"],
            "training_rows": record["training_rows"],
            "validation_rows": record["validation_rows"],
            "average_precision": record["metrics"]["average_precision"],
            "precision_at_100": record["metrics"]["daily_customer_precision_at_100"],
            **record["timings"],
            "mlflow_run_id": record["tracking"]["run_id"],
        }
    )
folds = pd.DataFrame(rows)
folds

## Fila diária e recall

Clientes se repetem entre dias e janelas. A fila retrospectiva usa máximo score por cliente/dia; não é autorização online. Recall descreve a fração de clientes com fraude alcançados em cada dia. Não tratar dispersão desses dias como intervalo de confiança.


In [ ]:
daily_rows = []
for receipt in sorted((STUDY_PATH / "models").glob("*/*/result.json")):
    predictions = pd.read_parquet(receipt.parent / "validation_predictions.parquet")
    predictions["day"] = predictions.TX_DATETIME.dt.strftime("%Y-%m-%d")
    customers = predictions.groupby(["day", "CUSTOMER_ID"]).TX_FRAUD.max()
    fraudulent = customers.groupby(level="day").sum()
    record = json.loads(receipt.read_text())
    for day in record["metrics"]["daily_customer_metrics"]:
        count = int(fraudulent.loc[day["date"]])
        found = day["fraudulent_customers_in_alerts"]
        daily_rows.append(
            {
                "candidate": receipt.parent.parent.name,
                "fold": record["fold_id"],
                **day,
                "fraudulent_customers": count,
                "fraudulent_customers_outside_alerts": count - found,
                "recall_at_100": found / count if count else None,
            }
        )
pd.DataFrame(daily_rows)

## Decisão registrada e limites

O autor informou 20 trials, nenhuma falha, 63 fits e `retain_reference`. O melhor
ganho médio de AP não atingiu o gate; nenhuma confirmação de candidato foi
executada. O [recibo do relatório](../../references/evidence/hgb_optuna_author_report_2026-10-09.json)
identifica a origem e os limites; este texto não é um output de execução da célula.

A referência fixa foi avaliada depois em 02–15/09, com diagnóstico e reamostragem
exploratória revisados. Essa etapa não transforma um trial inelegível em candidato
confirmatório. 16–30/09 permanece reservado para replay. Uma futura comparação
exige candidato elegível, hipótese, efeito relevante, unidade de inferência e
tratamento da dependência definidos antes de abrir outra janela preservada.
[Resultado e fechamento](../../docs/modeling/EVALUATION_PROTOCOL.md#resultado-nativo-da-reamostragem-e-fechamento).
